In [1]:
!pip -q install -U open_clip_torch
!pip -q install "transformers==4.41.2" "tokenizers==0.19.1" sentencepiece accelerate
!pip -q install evaluate rouge_score sacrebleu nltk tqdm pillow scikit-learn

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.0 MB/s eta 0:00:00
Torch: 2.11.0+cu128
CUDA available: True
G

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, zipfile, time

PROJECT_DIR = Path("/content/drive/MyDrive/Senior2_Medical_Captioning")

DEV_ZIP_DRIVE = PROJECT_DIR / "dev_caption.zip"
LOCAL_ZIP = Path("/content/dev_caption.zip")
LOCAL_DEV_DIR = Path("/content/dev_caption")
IMAGE_DIR = LOCAL_DEV_DIR / "images"

print("DEV_ZIP_DRIVE exists:", DEV_ZIP_DRIVE.exists())
print("DEV_ZIP_DRIVE size GB:", round(DEV_ZIP_DRIVE.stat().st_size / (1024**3), 2) if DEV_ZIP_DRIVE.exists() else None)

if not DEV_ZIP_DRIVE.exists():
    raise FileNotFoundError(f"Missing dev_caption.zip at:\n{DEV_ZIP_DRIVE}")

# Copy zip to local runtime for faster unzip/read
if not LOCAL_ZIP.exists():
    print("Copying dev_caption.zip to /content ...")
    start = time.time()
    shutil.copy2(DEV_ZIP_DRIVE, LOCAL_ZIP)
    print("Copy done in minutes:", round((time.time() - start) / 60, 2))
else:
    print("Local zip already exists:", LOCAL_ZIP)

# Unzip locally
if not IMAGE_DIR.exists():
    print("Unzipping dev_caption.zip to /content ...")
    start = time.time()
    !unzip -q -n "/content/dev_caption.zip" -d "/content"
    print("Unzip done in minutes:", round((time.time() - start) / 60, 2))
else:
    print("Images already extracted:", IMAGE_DIR)

print("IMAGE_DIR exists:", IMAGE_DIR.exists())

# Quick sample
sample_images = list(IMAGE_DIR.glob("*.jpg"))[:5]
print("Sample images:")
for p in sample_images:
    print(" -", p.name)

Mounted at /content/drive
DEV_ZIP_DRIVE exists: True
DEV_ZIP_DRIVE size GB: 8.88
Copying dev_caption.zip to /content ...
Copy done in minutes: 1.88
Unzipping dev_caption.zip to /content ...
Unzip done in minutes: 1.59
IMAGE_DIR exists: True
Sample images:
 - ImageCLEFmedical_Caption_2026_train_84223.jpg
 - ImageCLEFmedical_Caption_2026_train_53284.jpg
 - ImageCLEFmedical_Caption_2026_valid_17427.jpg
 - ImageCLEFmedical_Caption_2026_train_13904.jpg
 - ImageCLEFmedical_Caption_2026_train_47952.jpg


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import random, json, os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_DIR = PROJECT_DIR / "caption_generator_datasets_umls_terms"

TRAIN_CSV = DATASET_DIR / "gt_cui_terms_train.csv"
VALID_CSV = DATASET_DIR / "gt_cui_terms_valid.csv"

OUTPUT_DIR = PROJECT_DIR / "models" / "exp5_image_only_biomedclip_visual_prefix_t5"
FEATURE_DIR = OUTPUT_DIR / "precomputed_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_CSV exists:", TRAIN_CSV.exists())
print("VALID_CSV exists:", VALID_CSV.exists())

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

# Image-only: only ID and target_text are needed
train_df = train_df[["ID", "split", "target_text"]].copy()
valid_df = valid_df[["ID", "split", "target_text"]].copy()

# Same valid size used in previous comparisons
VALID_EVAL_SIZE = 3000
valid3000_df = valid_df.sample(n=VALID_EVAL_SIZE, random_state=SEED).reset_index(drop=True)

def add_image_path(df):
    df = df.copy()
    df["image_path"] = df["ID"].apply(lambda x: str(IMAGE_DIR / f"{x}.jpg"))
    return df

train_df = add_image_path(train_df)
valid3000_df = add_image_path(valid3000_df)

print("Train size:", len(train_df))
print("Valid3000 size:", len(valid3000_df))

# Verify image existence
def check_missing(df, name):
    missing = [p for p in df["image_path"].tolist() if not Path(p).exists()]
    print(f"{name} missing images:", len(missing))
    if len(missing) > 0:
        print("First missing:", missing[:5])
        raise FileNotFoundError(f"{name} has missing images.")

check_missing(train_df, "train")
check_missing(valid3000_df, "valid3000")

# Save metadata
train_meta_path = OUTPUT_DIR / "exp5_train_metadata.csv"
valid_meta_path = OUTPUT_DIR / "exp5_valid3000_metadata.csv"

train_df.to_csv(train_meta_path, index=False)
valid3000_df.to_csv(valid_meta_path, index=False)

print("Saved:")
print(train_meta_path)
print(valid_meta_path)

display(train_df.head())
display(valid3000_df.head())

TRAIN_CSV exists: True
VALID_CSV exists: True
Train size: 97364
Valid3000 size: 3000
train missing images: 0
valid3000 missing images: 0
Saved:
/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/exp5_train_metadata.csv
/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/exp5_valid3000_metadata.csv


,ID,split,target_text,image_path
0,ImageCLEFmedical_Caption_2026_train_0,train,Head CT demonstrating left parotiditis.,/content/dev_caption/images/ImageCLEFmedical_C...
1,ImageCLEFmedical_Caption_2026_train_1,train,Chest X-ray showing enlarged cardiac silhouett...,/content/dev_caption/images/ImageCLEFmedical_C...
2,ImageCLEFmedical_Caption_2026_train_2,train,CT chest axial view showing a huge ascending a...,/content/dev_caption/images/ImageCLEFmedical_C...
3,ImageCLEFmedical_Caption_2026_train_3,train,Acquired renal cysts in end-stage renal failur...,/content/dev_caption/images/ImageCLEFmedical_C...
4,ImageCLEFmedical_Caption_2026_train_4,train,Computed tomography (CT) shows floating thromb...,/content/dev_caption/images/ImageCLEFmedical_C...


,ID,split,target_text,image_path
0,ImageCLEFmedical_Caption_2026_valid_12658,valid,Ultra-high-frequency ultrasound (48 MHz probe)...,/content/dev_caption/images/ImageCLEFmedical_C...
1,ImageCLEFmedical_Caption_2026_valid_17336,valid,A midesophageal long axis view zoomed up on th...,/content/dev_caption/images/ImageCLEFmedical_C...
2,ImageCLEFmedical_Caption_2026_valid_18790,valid,General aspect of the uterus and of the cervix...,/content/dev_caption/images/ImageCLEFmedical_C...
3,ImageCLEFmedical_Caption_2026_valid_12815,valid,computed tomography of the chest with a white ...,/content/dev_caption/images/ImageCLEFmedical_C...
4,ImageCLEFmedical_Caption_2026_valid_12666,valid,Lateral view of the skull showing multiple pun...,/content/dev_caption/images/ImageCLEFmedical_C...


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
import open_clip
from transformers import AutoTokenizer, T5ForConditionalGeneration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available()

BIOMEDCLIP_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

EXP4_BEST_MODEL = (
    PROJECT_DIR
    / "models"
    / "exp4_mixed_gt_pred_micro_terms_t5_base_safe5"
    / "best_model"
)

print("DEVICE:", DEVICE)
print("USE_BF16:", USE_BF16)
print("EXP4_BEST_MODEL exists:", EXP4_BEST_MODEL.exists())

if not EXP4_BEST_MODEL.exists():
    raise FileNotFoundError(f"Exp4 best model not found:\n{EXP4_BEST_MODEL}")

print("\nLoading BiomedCLIP...")

try:
    biomedclip_model, _, biomedclip_preprocess = open_clip.create_model_and_transforms(BIOMEDCLIP_NAME)
except Exception as e:
    print("create_model_and_transforms failed, trying create_model_from_pretrained...")
    print("Error:", e)
    from open_clip import create_model_from_pretrained
    biomedclip_model, biomedclip_preprocess = create_model_from_pretrained(BIOMEDCLIP_NAME)

biomedclip_model = biomedclip_model.to(DEVICE)
biomedclip_model.eval()

for p in biomedclip_model.parameters():
    p.requires_grad = False

print("BiomedCLIP loaded.")

# Test one image and detect feature dimension
test_image_path = valid3000_df["image_path"].iloc[0]
img = Image.open(test_image_path).convert("RGB")
img_tensor = biomedclip_preprocess(img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    image_feat = biomedclip_model.encode_image(img_tensor)
    image_feat = F.normalize(image_feat.float(), dim=-1)

IMAGE_FEATURE_DIM = image_feat.shape[-1]
print("BiomedCLIP image feature shape:", image_feat.shape)
print("IMAGE_FEATURE_DIM:", IMAGE_FEATURE_DIM)

print("\nLoading Exp4-initialized T5...")
tokenizer = AutoTokenizer.from_pretrained(EXP4_BEST_MODEL)
t5_model = T5ForConditionalGeneration.from_pretrained(EXP4_BEST_MODEL)

t5_model.config.use_cache = False
t5_model.gradient_checkpointing_enable()

print("T5 loaded.")
print("T5 d_model:", t5_model.config.d_model)

DEVICE: cuda
USE_BF16: True
EXP4_BEST_MODEL exists: True

Loading BiomedCLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

BiomedCLIP loaded.
BiomedCLIP image feature shape: torch.Size([1, 512])
IMAGE_FEATURE_DIM: 512

Loading Exp4-initialized T5...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


T5 loaded.
T5 d_model: 768


In [5]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

class ImageFeatureDataset(Dataset):
    def __init__(self, df, preprocess):
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        image = Image.open(image_path).convert("RGB")
        pixel_values = self.preprocess(image)
        return {
            "pixel_values": pixel_values,
            "ID": row["ID"]
        }

def collate_images(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    ids = [b["ID"] for b in batch]
    return pixel_values, ids

def precompute_image_features(df, split_name, batch_size=128, num_workers=2):
    out_npy = FEATURE_DIR / f"{split_name}_biomedclip_features.npy"
    out_ids = FEATURE_DIR / f"{split_name}_ids.csv"

    if out_npy.exists() and out_ids.exists():
        existing = np.load(out_npy, mmap_mode="r")
        if existing.shape[0] == len(df) and existing.shape[1] == IMAGE_FEATURE_DIM:
            print(f"[SKIP] Features already exist for {split_name}: {existing.shape}")
            return out_npy, out_ids

    dataset = ImageFeatureDataset(df, biomedclip_preprocess)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collate_images
    )

    features_memmap = np.lib.format.open_memmap(
        out_npy,
        dtype=np.float16,
        mode="w+",
        shape=(len(df), IMAGE_FEATURE_DIM)
    )

    all_ids = []
    cursor = 0

    biomedclip_model.eval()

    print(f"Precomputing features for {split_name} | n={len(df)}")

    for pixel_values, ids in tqdm(loader, desc=f"BiomedCLIP features: {split_name}"):
        pixel_values = pixel_values.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                feats = biomedclip_model.encode_image(pixel_values)
            feats = F.normalize(feats.float(), dim=-1)

        feats_np = feats.detach().cpu().numpy().astype(np.float16)
        bs = feats_np.shape[0]

        features_memmap[cursor:cursor+bs] = feats_np
        all_ids.extend(ids)
        cursor += bs

    pd.DataFrame({"ID": all_ids}).to_csv(out_ids, index=False)

    print(f"Saved features: {out_npy}")
    print(f"Saved IDs: {out_ids}")
    print("Shape:", features_memmap.shape)

    return out_npy, out_ids

train_feat_path, train_ids_path = precompute_image_features(
    train_df,
    "train",
    batch_size=128,
    num_workers=2
)

valid_feat_path, valid_ids_path = precompute_image_features(
    valid3000_df,
    "valid3000",
    batch_size=128,
    num_workers=2
)

Precomputing features for train | n=97364


BiomedCLIP features: train:   0%|          | 0/761 [00:00<?, ?it/s]

Saved features: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/precomputed_features/train_biomedclip_features.npy
Saved IDs: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/precomputed_features/train_ids.csv
Shape: (97364, 512)
Precomputing features for valid3000 | n=3000


BiomedCLIP features: valid3000:   0%|          | 0/24 [00:00<?, ?it/s]

Saved features: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/precomputed_features/valid3000_biomedclip_features.npy
Saved IDs: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/precomputed_features/valid3000_ids.csv
Shape: (3000, 512)


In [6]:
MAX_TARGET_LEN = 96

def pretokenize_targets(df, split_name):
    out_labels = FEATURE_DIR / f"{split_name}_labels_maxlen{MAX_TARGET_LEN}.npy"

    if out_labels.exists():
        arr = np.load(out_labels, mmap_mode="r")
        if arr.shape[0] == len(df):
            print(f"[SKIP] Labels already exist for {split_name}: {arr.shape}")
            return out_labels

    texts = df["target_text"].fillna("").astype(str).tolist()

    encoded = tokenizer(
        texts,
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="np"
    )

    labels = encoded["input_ids"].astype(np.int64)
    labels[labels == tokenizer.pad_token_id] = -100

    np.save(out_labels, labels)
    print(f"Saved labels: {out_labels} | shape={labels.shape}")

    return out_labels

train_labels_path = pretokenize_targets(train_df, "train")
valid_labels_path = pretokenize_targets(valid3000_df, "valid3000")


class PrecomputedFeatureCaptionDataset(Dataset):
    def __init__(self, df, feature_path, label_path):
        self.df = df.reset_index(drop=True)
        self.features = np.load(feature_path, mmap_mode="r")
        self.labels = np.load(label_path, mmap_mode="r")

        assert len(self.df) == self.features.shape[0]
        assert len(self.df) == self.labels.shape[0]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_features = torch.tensor(self.features[idx], dtype=torch.float32)
        labels = torch.tensor(self.labels[idx], dtype=torch.long)

        return {
            "image_features": image_features,
            "labels": labels,
            "ID": self.df.iloc[idx]["ID"],
            "target_text": self.df.iloc[idx]["target_text"]
        }

def collate_feature_caption(batch):
    image_features = torch.stack([b["image_features"] for b in batch])
    labels = torch.stack([b["labels"] for b in batch])
    ids = [b["ID"] for b in batch]
    targets = [b["target_text"] for b in batch]

    return {
        "image_features": image_features,
        "labels": labels,
        "ID": ids,
        "target_text": targets
    }

train_dataset = PrecomputedFeatureCaptionDataset(train_df, train_feat_path, train_labels_path)
valid_dataset = PrecomputedFeatureCaptionDataset(valid3000_df, valid_feat_path, valid_labels_path)

print("Train dataset:", len(train_dataset))
print("Valid dataset:", len(valid_dataset))

sample = train_dataset[0]
print("Sample feature shape:", sample["image_features"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Sample target:", sample["target_text"])

Saved labels: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/precomputed_features/train_labels_maxlen96.npy | shape=(97364, 96)
Saved labels: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/precomputed_features/valid3000_labels_maxlen96.npy | shape=(3000, 96)
Train dataset: 97364
Valid dataset: 3000
Sample feature shape: torch.Size([512])
Sample labels shape: torch.Size([96])
Sample target: Head CT demonstrating left parotiditis.


In [7]:
class ImageOnlyVisualPrefixT5(nn.Module):
    def __init__(self, t5, image_feature_dim, prefix_len=16, dropout=0.1):
        super().__init__()

        self.t5 = t5
        self.prefix_len = prefix_len
        self.d_model = t5.config.d_model

        self.visual_projector = nn.Sequential(
            nn.LayerNorm(image_feature_dim),
            nn.Linear(image_feature_dim, self.d_model * prefix_len),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.d_model * prefix_len, self.d_model * prefix_len)
        )

    def build_prefix(self, image_features):
        prefix = self.visual_projector(image_features)
        prefix = prefix.view(image_features.size(0), self.prefix_len, self.d_model)

        # Match T5 dtype
        t5_dtype = next(self.t5.parameters()).dtype
        prefix = prefix.to(dtype=t5_dtype)

        attention_mask = torch.ones(
            prefix.size(0),
            prefix.size(1),
            dtype=torch.long,
            device=prefix.device
        )

        return prefix, attention_mask

    def forward(self, image_features, labels=None):
        inputs_embeds, attention_mask = self.build_prefix(image_features)

        return self.t5(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels
        )

    @torch.no_grad()
    def generate_from_features(self, image_features, **generate_kwargs):
        inputs_embeds, attention_mask = self.build_prefix(image_features)

        return self.t5.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            **generate_kwargs
        )


PREFIX_LEN = 16
DROPOUT = 0.1

model = ImageOnlyVisualPrefixT5(
    t5=t5_model,
    image_feature_dim=IMAGE_FEATURE_DIM,
    prefix_len=PREFIX_LEN,
    dropout=DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Prefix length:", PREFIX_LEN)

Total parameters: 382,226,944
Trainable parameters: 382,226,944
Prefix length: 16


In [8]:
from torch.utils.data import DataLoader, Subset
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
import math
from tqdm.auto import tqdm

SANITY_BATCH_SIZE = 16
SANITY_STEPS = 50

sanity_subset = Subset(train_dataset, list(range(512)))

sanity_loader = DataLoader(
    sanity_subset,
    batch_size=SANITY_BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_feature_caption
)

sanity_optimizer = AdamW(
    [
        {"params": model.visual_projector.parameters(), "lr": 1e-4},
        {"params": model.t5.parameters(), "lr": 1e-5},
    ],
    weight_decay=0.01
)

model.train()
losses = []

print("Starting quick sanity check...")

for step, batch in enumerate(tqdm(sanity_loader, total=SANITY_STEPS)):
    if step >= SANITY_STEPS:
        break

    image_features = batch["image_features"].to(DEVICE, non_blocking=True)
    labels = batch["labels"].to(DEVICE, non_blocking=True)

    sanity_optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        outputs = model(image_features=image_features, labels=labels)
        loss = outputs.loss

    if torch.isnan(loss):
        raise ValueError("NaN loss detected during sanity check.")

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    sanity_optimizer.step()

    losses.append(float(loss.detach().cpu()))

print("Sanity check finished.")
print("First loss:", losses[0])
print("Last loss:", losses[-1])
print("Mean loss:", sum(losses) / len(losses))

Starting quick sanity check...


  0%|          | 0/50 [00:00<?, ?it/s]

Sanity check finished.
First loss: 5.859881401062012
Last loss: 3.244673252105713
Mean loss: 3.454803913831711


In [9]:
# Cell 8.5 — Reset Exp5 model fresh before full training
# This removes the small updates from the sanity check and starts full training cleanly.

import gc
import torch
import torch.nn as nn
from transformers import AutoTokenizer, T5ForConditionalGeneration
from pathlib import Path

# Clean GPU memory
try:
    del model
except:
    pass

try:
    del t5_model
except:
    pass

gc.collect()
torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available()

EXP4_BEST_MODEL = (
    PROJECT_DIR
    / "models"
    / "exp4_mixed_gt_pred_micro_terms_t5_base_safe5"
    / "best_model"
)

print("DEVICE:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("EXP4_BEST_MODEL exists:", EXP4_BEST_MODEL.exists())

if not EXP4_BEST_MODEL.exists():
    raise FileNotFoundError(f"Exp4 best model not found:\n{EXP4_BEST_MODEL}")

# Reload tokenizer and T5 fresh from Exp4 best model
print("\nReloading tokenizer and Exp4-initialized T5...")
tokenizer = AutoTokenizer.from_pretrained(EXP4_BEST_MODEL)
t5_model = T5ForConditionalGeneration.from_pretrained(EXP4_BEST_MODEL)

t5_model.config.use_cache = False
t5_model.gradient_checkpointing_enable()

print("T5 loaded fresh.")
print("T5 d_model:", t5_model.config.d_model)


# Define model again, in case runtime lost the class
class ImageOnlyVisualPrefixT5(nn.Module):
    def __init__(self, t5, image_feature_dim, prefix_len=16, dropout=0.1):
        super().__init__()

        self.t5 = t5
        self.prefix_len = prefix_len
        self.d_model = t5.config.d_model

        self.visual_projector = nn.Sequential(
            nn.LayerNorm(image_feature_dim),
            nn.Linear(image_feature_dim, self.d_model * prefix_len),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.d_model * prefix_len, self.d_model * prefix_len)
        )

    def build_prefix(self, image_features):
        prefix = self.visual_projector(image_features)
        prefix = prefix.view(image_features.size(0), self.prefix_len, self.d_model)

        t5_dtype = next(self.t5.parameters()).dtype
        prefix = prefix.to(dtype=t5_dtype)

        attention_mask = torch.ones(
            prefix.size(0),
            prefix.size(1),
            dtype=torch.long,
            device=prefix.device
        )

        return prefix, attention_mask

    def forward(self, image_features, labels=None):
        inputs_embeds, attention_mask = self.build_prefix(image_features)

        return self.t5(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels
        )

    @torch.no_grad()
    def generate_from_features(self, image_features, **generate_kwargs):
        inputs_embeds, attention_mask = self.build_prefix(image_features)

        return self.t5.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            **generate_kwargs
        )


# Rebuild model fresh
PREFIX_LEN = 16
DROPOUT = 0.1

model = ImageOnlyVisualPrefixT5(
    t5=t5_model,
    image_feature_dim=IMAGE_FEATURE_DIM,
    prefix_len=PREFIX_LEN,
    dropout=DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nFresh Exp5 model is ready.")
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Prefix length:", PREFIX_LEN)
print("Image feature dim:", IMAGE_FEATURE_DIM)

DEVICE: cuda
GPU: NVIDIA A100-SXM4-80GB
EXP4_BEST_MODEL exists: True

Reloading tokenizer and Exp4-initialized T5...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


T5 loaded fresh.
T5 d_model: 768

Fresh Exp5 model is ready.
Total parameters: 382,226,944
Trainable parameters: 382,226,944
Prefix length: 16
Image feature dim: 512


In [10]:
BATCH_SIZE = 32
GRAD_ACCUM = 1
EPOCHS = 2

ADAPTER_LR = 1e-4
T5_LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_feature_caption
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_feature_caption
)

optimizer = AdamW(
    [
        {"params": model.visual_projector.parameters(), "lr": ADAPTER_LR},
        {"params": model.t5.parameters(), "lr": T5_LR},
    ],
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_train_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_train_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_train_steps
)

print("Steps per epoch:", steps_per_epoch)
print("Total train steps:", total_train_steps)
print("Warmup steps:", warmup_steps)


@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    losses = []

    for batch in tqdm(loader, desc="Validation loss"):
        image_features = batch["image_features"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16):
            outputs = model(image_features=image_features, labels=labels)
            loss = outputs.loss

        losses.append(float(loss.detach().cpu()))

    return float(np.mean(losses))


def save_exp5_model(model, save_dir):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    model.t5.save_pretrained(save_dir / "t5")
    tokenizer.save_pretrained(save_dir / "t5")

    torch.save(
        {
            "visual_projector_state_dict": model.visual_projector.state_dict(),
            "image_feature_dim": IMAGE_FEATURE_DIM,
            "prefix_len": PREFIX_LEN,
            "dropout": DROPOUT,
            "base_t5": str(EXP4_BEST_MODEL),
        },
        save_dir / "visual_prefix_adapter.pt"
    )

    config = {
        "experiment": "Exp5 Image-only BiomedCLIP Visual Prefix + Exp4-initialized T5",
        "image_encoder": BIOMEDCLIP_NAME,
        "image_encoder_frozen": True,
        "t5_initialized_from": str(EXP4_BEST_MODEL),
        "prefix_len": PREFIX_LEN,
        "image_feature_dim": IMAGE_FEATURE_DIM,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "epochs": EPOCHS,
        "adapter_lr": ADAPTER_LR,
        "t5_lr": T5_LR,
        "max_target_len": MAX_TARGET_LEN,
        "valid_eval_size": VALID_EVAL_SIZE,
        "seed": SEED,
    }

    with open(save_dir / "exp5_config.json", "w") as f:
        json.dump(config, f, indent=2)

    print("Saved model to:", save_dir)


BEST_DIR = OUTPUT_DIR / "best_model"
LAST_DIR = OUTPUT_DIR / "last_model"
history = []
best_val_loss = float("inf")

global_step = 0

print("Starting full training...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_losses = []

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress, start=1):
        image_features = batch["image_features"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16):
            outputs = model(image_features=image_features, labels=labels)
            loss = outputs.loss / GRAD_ACCUM

        if torch.isnan(loss):
            raise ValueError(f"NaN loss detected at epoch={epoch}, step={step}")

        loss.backward()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        running_losses.append(float(loss.detach().cpu()) * GRAD_ACCUM)

        progress.set_postfix({
            "loss": f"{np.mean(running_losses[-50:]):.4f}",
            "lr_t5": optimizer.param_groups[1]["lr"]
        })

    train_loss = float(np.mean(running_losses))
    val_loss = evaluate_loss(model, valid_loader)

    row = {
        "epoch": epoch,
        "global_step": global_step,
        "train_loss": train_loss,
        "val_loss": val_loss,
    }
    history.append(row)

    print("\nEpoch summary:", row)

    pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

    save_exp5_model(model, LAST_DIR)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        print("New best val_loss:", best_val_loss)
        save_exp5_model(model, BEST_DIR)

print("Training finished.")
print("Best val_loss:", best_val_loss)

Steps per epoch: 3043
Total train steps: 6086
Warmup steps: 304
Starting full training...


Epoch 1/2:   0%|          | 0/3043 [00:00<?, ?it/s]

Validation loss:   0%|          | 0/94 [00:00<?, ?it/s]


Epoch summary: {'epoch': 1, 'global_step': 3043, 'train_loss': 2.826091007304121, 'val_loss': 2.58045782941453}
Saved model to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/last_model
New best val_loss: 2.58045782941453
Saved model to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/best_model


Epoch 2/2:   0%|          | 0/3043 [00:00<?, ?it/s]

Validation loss:   0%|          | 0/94 [00:00<?, ?it/s]


Epoch summary: {'epoch': 2, 'global_step': 6086, 'train_loss': 2.729995578360957, 'val_loss': 2.55249304720696}
Saved model to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/last_model
New best val_loss: 2.55249304720696
Saved model to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/best_model
Training finished.
Best val_loss: 2.55249304720696


In [11]:
import evaluate
import sacrebleu

GEN_BATCH_SIZE = 32
GEN_MAX_LEN = 96
GEN_NUM_BEAMS = 2

model.eval()

generation_loader = DataLoader(
    valid_dataset,
    batch_size=GEN_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_feature_caption
)

all_ids = []
all_targets = []
all_predictions = []

print("Generating captions for valid3000...")

for batch in tqdm(generation_loader, desc="Generating"):
    image_features = batch["image_features"].to(DEVICE, non_blocking=True)

    generated_ids = model.generate_from_features(
        image_features=image_features,
        max_length=GEN_MAX_LEN,
        num_beams=GEN_NUM_BEAMS,
        early_stopping=True
    )

    preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    all_ids.extend(batch["ID"])
    all_targets.extend(batch["target_text"])
    all_predictions.extend(preds)

pred_df = pd.DataFrame({
    "ID": all_ids,
    "experiment": "Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5",
    "setting": "Image only",
    "target_text": all_targets,
    "generated_text": all_predictions
})

pred_path = OUTPUT_DIR / "exp5_image_only_valid3000_predictions.csv"
pred_df.to_csv(pred_path, index=False)

print("Saved predictions to:")
print(pred_path)

display(pred_df.head(10))

print("Computing ROUGE...")
rouge_metric = evaluate.load("rouge")

rouge_scores = rouge_metric.compute(
    predictions=all_predictions,
    references=all_targets,
    use_stemmer=True
)

print("Computing BLEU...")
bleu_score = sacrebleu.corpus_bleu(
    all_predictions,
    [all_targets]
).score

metrics = {
    "experiment": "Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5",
    "setting": "Image only",
    "n_samples": len(pred_df),
    "rouge1": float(rouge_scores["rouge1"]),
    "rouge2": float(rouge_scores["rouge2"]),
    "rougeL": float(rouge_scores["rougeL"]),
    "bleu": float(bleu_score),
    "best_val_loss": float(best_val_loss),
    "predictions_file": str(pred_path)
}

metrics_path = OUTPUT_DIR / "exp5_image_only_valid3000_rouge_bleu_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics:")
for k, v in metrics.items():
    print(k, ":", v)

print("\nSaved metrics to:")
print(metrics_path)

Generating captions for valid3000...


Generating:   0%|          | 0/94 [00:00<?, ?it/s]

Saved predictions to:
/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/exp5_image_only_valid3000_predictions.csv


,ID,experiment,setting,target_text,generated_text
0,ImageCLEFmedical_Caption_2026_valid_12658,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,Ultra-high-frequency ultrasound (48 MHz probe)...,Ultrasound image of a solitary solitary solita...
1,ImageCLEFmedical_Caption_2026_valid_17336,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,A midesophageal long axis view zoomed up on th...,Transesophageal echocardiogram showing a large...
2,ImageCLEFmedical_Caption_2026_valid_18790,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,General aspect of the uterus and of the cervix...,Ultrasound of the abdomen and pelvis showing a...
3,ImageCLEFmedical_Caption_2026_valid_12815,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,computed tomography of the chest with a white ...,Axial CT image of the chest showing a large ri...
4,ImageCLEFmedical_Caption_2026_valid_12666,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,Lateral view of the skull showing multiple pun...,Lateral X-ray of the skull.
5,ImageCLEFmedical_Caption_2026_valid_11890,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,Preoperative craniocaudal view of both shoulde...,X-ray of the pelvis.
6,ImageCLEFmedical_Caption_2026_valid_3001,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,CT Abdomen and Pelvis with contrast (Axial Vie...,Computed tomography of the abdomen and pelvis ...
7,ImageCLEFmedical_Caption_2026_valid_8202,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,Chest CT-gross right pleural effusion (white a...,Computed tomography (CT) scan of the chest sho...
8,ImageCLEFmedical_Caption_2026_valid_14362,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,The preoperative abdominopelvic contrast CT im...,Computed tomography (CT) scan of the abdomen a...
9,ImageCLEFmedical_Caption_2026_valid_14768,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,Angiography of the left vertebral artery. Trau...,Angiography of the left internal carotid arter...


Computing ROUGE...


Computing BLEU...

Metrics:
experiment : Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5
setting : Image only
n_samples : 3000
rouge1 : 0.2401045312227293
rouge2 : 0.08056657612442697
rougeL : 0.20699749429818054
bleu : 2.8757232877880985
best_val_loss : 2.55249304720696
predictions_file : /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/exp5_image_only_valid3000_predictions.csv

Saved metrics to:
/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/exp5_image_only_valid3000_rouge_bleu_metrics.json


In [12]:
!pip -q install "bert-score==0.3.13"
!pip -q install git+https://github.com/lucadiliello/bleurt-pytorch.git

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Torch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json, os, gc
from tqdm.auto import tqdm

PROJECT_DIR = Path("/content/drive/MyDrive/Senior2_Medical_Captioning")

EXP5_DIR = PROJECT_DIR / "models" / "exp5_image_only_biomedclip_visual_prefix_t5"

PRED_PATH = EXP5_DIR / "exp5_image_only_valid3000_predictions.csv"
MAPPING_PATH = PROJECT_DIR / "cui_to_umls_terms.csv"

METRIC_OUTPUT_DIR = EXP5_DIR / "additional_metrics"
METRIC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PRED_PATH exists:", PRED_PATH.exists())
print("MAPPING_PATH exists:", MAPPING_PATH.exists())

if not PRED_PATH.exists():
    raise FileNotFoundError(PRED_PATH)

if not MAPPING_PATH.exists():
    raise FileNotFoundError(MAPPING_PATH)

df = pd.read_csv(PRED_PATH)

print("Exp5 predictions shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())


# ---------------------------
# Text preprocessing
# ---------------------------
def preprocess_caption_for_metric(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_for_match(text):
    text = preprocess_caption_for_metric(text).lower()
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ---------------------------
# Load CUI → UMLS term mapping
# ---------------------------
mapping_df = pd.read_csv(MAPPING_PATH)

print("\nMapping columns:", list(mapping_df.columns))
display(mapping_df.head())

# Robustly detect columns
possible_cui_cols = ["cui", "CUI", "concept_id", "umls_cui"]
possible_term_cols = ["term", "name", "preferred_term", "umls_term", "TERM", "label"]

cui_col = None
term_col = None

for c in mapping_df.columns:
    if c in possible_cui_cols or "cui" in c.lower():
        cui_col = c
        break

for c in mapping_df.columns:
    if c in possible_term_cols or "term" in c.lower() or "name" in c.lower():
        term_col = c
        break

if cui_col is None or term_col is None:
    raise ValueError(f"Could not detect CUI/term columns. Columns: {list(mapping_df.columns)}")

print("Detected CUI column:", cui_col)
print("Detected term column:", term_col)

mapping_df = mapping_df[[cui_col, term_col]].dropna()
mapping_df[cui_col] = mapping_df[cui_col].astype(str).str.strip()
mapping_df[term_col] = mapping_df[term_col].astype(str).str.strip()

mapping_df = mapping_df[mapping_df[cui_col].str.match(r"^C\d{7}$", na=False)]

# Known correction from earlier experiments
cui_to_term = dict(zip(mapping_df[cui_col], mapping_df[term_col]))
cui_to_term["C0043262"] = "Wrist"

print("Number of CUI terms:", len(cui_to_term))


# ---------------------------
# Build aliases for dictionary-based UMLS matching
# ---------------------------
stop_terms = {
    "image", "images", "finding", "findings", "patient", "patients",
    "case", "view", "left", "right", "large", "small", "normal",
    "abnormal", "medical", "disease", "structure"
}

manual_aliases = {
    "computed tomography": ["ct", "ct scan", "computed tomographic"],
    "x ray": ["xray", "x-ray", "radiograph", "radiography"],
    "magnetic resonance imaging": ["mri", "mr imaging"],
    "ultrasonography": ["ultrasound", "sonography", "us"],
}

aliases = []

for cui, term in cui_to_term.items():
    norm_term = normalize_for_match(term)

    if len(norm_term) < 3:
        continue

    if norm_term in stop_terms:
        continue

    aliases.append((norm_term, cui))

    # Add manual aliases when term contains known expressions
    for key, vals in manual_aliases.items():
        if key in norm_term:
            for v in vals:
                aliases.append((normalize_for_match(v), cui))

# remove duplicates and sort by longer terms first
aliases = list(set(aliases))
aliases = sorted(aliases, key=lambda x: len(x[0]), reverse=True)

FAST_UMLS_ALIASES = []
for alias, cui in aliases:
    alias = normalize_for_match(alias)
    if alias and len(alias) >= 3:
        FAST_UMLS_ALIASES.append((" " + alias + " ", cui))

FAST_UMLS_ALIASES = list(set(FAST_UMLS_ALIASES))
FAST_UMLS_ALIASES = sorted(FAST_UMLS_ALIASES, key=lambda x: len(x[0]), reverse=True)

print("Fast UMLS aliases:", len(FAST_UMLS_ALIASES))

PRED_PATH exists: True
MAPPING_PATH exists: True
Exp5 predictions shape: (3000, 5)
Columns: ['ID', 'experiment', 'setting', 'target_text', 'generated_text']


,ID,experiment,setting,target_text,generated_text
0,ImageCLEFmedical_Caption_2026_valid_12658,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,Ultra-high-frequency ultrasound (48 MHz probe)...,Ultrasound image of a solitary solitary solita...
1,ImageCLEFmedical_Caption_2026_valid_17336,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,A midesophageal long axis view zoomed up on th...,Transesophageal echocardiogram showing a large...
2,ImageCLEFmedical_Caption_2026_valid_18790,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,General aspect of the uterus and of the cervix...,Ultrasound of the abdomen and pelvis showing a...
3,ImageCLEFmedical_Caption_2026_valid_12815,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,computed tomography of the chest with a white ...,Axial CT image of the chest showing a large ri...
4,ImageCLEFmedical_Caption_2026_valid_12666,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,Lateral view of the skull showing multiple pun...,Lateral X-ray of the skull.



Mapping columns: ['CUI', 'name', 'semantic_types', 'status_code', 'error']


,CUI,name,semantic_types,status_code,error
0,C0000726,Abdomen,Body Location or Region,200,NaN
1,C0000741,Abducens nerve structure,"Body Part, Organ, or Organ Component",200,NaN
2,C0000833,Abscess,Disease or Syndrome,200,NaN
3,C0000846,Agenesis,Congenital Abnormality,200,NaN
4,C0000962,Bone structure of acetabulum,"Body Part, Organ, or Organ Component",200,NaN


Detected CUI column: CUI
Detected term column: name
Number of CUI terms: 2639
Fast UMLS aliases: 2662


In [14]:
import torch
import transformers.pytorch_utils as ptu

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Patch for bleurt_pytorch compatibility if needed
if not hasattr(ptu, "find_pruneable_heads_and_indices"):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - set(already_pruned_heads)
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask), dtype=torch.long)[mask].clone().detach()
        return heads, index

    ptu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices

from bert_score import BERTScorer
from bleurt_pytorch import BleurtForSequenceClassification, BleurtTokenizer

BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"
BLEURT_MODEL = "lucadiliello/BLEURT-20"

BERTSCORE_BATCH_SIZE = 4
BLEURT_BATCH_SIZE = 16

references_for_idf = (
    df["target_text"]
    .fillna("")
    .astype(str)
    .apply(preprocess_caption_for_metric)
    .tolist()
)

print("Loading BERTScorer with IDF references...")
bertscorer = BERTScorer(
    model_type=BERTSCORE_MODEL,
    idf=True,
    idf_sents=references_for_idf,
    device=DEVICE,
    batch_size=BERTSCORE_BATCH_SIZE,
    rescale_with_baseline=False
)

print("BERTScorer loaded.")

print("Loading BLEURT...")
bleurt_tokenizer = BleurtTokenizer.from_pretrained(BLEURT_MODEL)
bleurt_model = BleurtForSequenceClassification.from_pretrained(BLEURT_MODEL)
bleurt_model = bleurt_model.to(DEVICE)
bleurt_model.eval()

print("BLEURT loaded.")

Loading BERTScorer with IDF references...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BleurtSPTokenizer'. 
The class this function is called from is 'BertTokenizer'.


spm.model:   0%|          | 0.00/4.70M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

BLEURT loaded.


In [15]:
from functools import lru_cache

@lru_cache(maxsize=100000)
def extract_umls_concepts_from_caption_fast(text):
    norm_text = " " + normalize_for_match(text) + " "
    found = set()

    for alias_padded, cui in FAST_UMLS_ALIASES:
        if alias_padded in norm_text:
            found.add(cui)

    return tuple(sorted(found))


def concept_prf(pred_set, ref_set):
    pred_set = set(pred_set)
    ref_set = set(ref_set)

    if len(pred_set) == 0 and len(ref_set) == 0:
        return np.nan, np.nan, np.nan

    if len(pred_set) == 0 and len(ref_set) > 0:
        return 0.0, 0.0, 0.0

    if len(pred_set) > 0 and len(ref_set) == 0:
        return 0.0, np.nan, 0.0

    tp = len(pred_set.intersection(ref_set))
    precision = tp / len(pred_set) if len(pred_set) > 0 else 0.0
    recall = tp / len(ref_set) if len(ref_set) > 0 else 0.0

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1


def compute_fast_umls_metrics(dfp):
    pred_sets = []
    ref_sets = []
    precisions = []
    recalls = []
    f1s = []

    preds = dfp["generated_text"].fillna("").astype(str).tolist()
    refs = dfp["target_text"].fillna("").astype(str).tolist()

    for pred, ref in tqdm(list(zip(preds, refs)), total=len(dfp), desc="Fast UMLS F1"):
        pred_set = set(extract_umls_concepts_from_caption_fast(pred))
        ref_set = set(extract_umls_concepts_from_caption_fast(ref))

        p, r, f1 = concept_prf(pred_set, ref_set)

        pred_sets.append(";".join(sorted(pred_set)))
        ref_sets.append(";".join(sorted(ref_set)))
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    return pred_sets, ref_sets, precisions, recalls, f1s


def compute_bertscore(predictions, references):
    predictions_clean = [preprocess_caption_for_metric(x) for x in predictions]
    references_clean = [preprocess_caption_for_metric(x) for x in references]

    P, R, F1 = bertscorer.score(
        predictions_clean,
        references_clean,
        batch_size=BERTSCORE_BATCH_SIZE,
        verbose=True
    )

    return (
        P.detach().cpu().numpy(),
        R.detach().cpu().numpy(),
        F1.detach().cpu().numpy()
    )


@torch.no_grad()
def compute_bleurt(predictions, references):
    predictions_clean = [preprocess_caption_for_metric(x) for x in predictions]
    references_clean = [preprocess_caption_for_metric(x) for x in references]

    scores = []

    for i in tqdm(range(0, len(predictions_clean), BLEURT_BATCH_SIZE), desc="BLEURT"):
        batch_preds = predictions_clean[i:i + BLEURT_BATCH_SIZE]
        batch_refs = references_clean[i:i + BLEURT_BATCH_SIZE]

        inputs = bleurt_tokenizer(
            batch_refs,
            batch_preds,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = bleurt_model(**inputs)
        batch_scores = outputs.logits.squeeze(-1).detach().cpu().numpy().tolist()
        scores.extend(batch_scores)

    return np.array(scores)


print("Starting additional metrics for Exp5...")

df_metrics = df.copy()

predictions = df_metrics["generated_text"].fillna("").astype(str).tolist()
references = df_metrics["target_text"].fillna("").astype(str).tolist()

# 1. Dictionary-based UMLS Concept F1
print("\nComputing Dictionary-based UMLS Concept F1...")
pred_sets, ref_sets, umls_p, umls_r, umls_f1 = compute_fast_umls_metrics(df_metrics)

df_metrics["umls_pred_cuis_extracted"] = pred_sets
df_metrics["umls_ref_cuis_extracted"] = ref_sets
df_metrics["dict_umls_precision"] = umls_p
df_metrics["dict_umls_recall"] = umls_r
df_metrics["dict_umls_f1"] = umls_f1

# 2. BERTScore
print("\nComputing BERTScore...")
bs_p, bs_r, bs_f1 = compute_bertscore(predictions, references)

df_metrics["bertscore_precision"] = bs_p
df_metrics["bertscore_recall"] = bs_r
df_metrics["bertscore_f1"] = bs_f1

# 3. BLEURT
print("\nComputing BLEURT...")
bleurt_scores = compute_bleurt(predictions, references)

df_metrics["bleurt"] = bleurt_scores

# Save per-sample metrics
per_sample_path = METRIC_OUTPUT_DIR / "exp5_image_only_valid3000_three_additional_metrics_per_sample.csv"
df_metrics.to_csv(per_sample_path, index=False)

# Summary row
summary = {
    "experiment": "Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5",
    "setting": "Image only",
    "n_samples": len(df_metrics),

    "bertscore_precision": float(np.nanmean(df_metrics["bertscore_precision"])),
    "bertscore_recall": float(np.nanmean(df_metrics["bertscore_recall"])),
    "bertscore_f1": float(np.nanmean(df_metrics["bertscore_f1"])),

    "bleurt": float(np.nanmean(df_metrics["bleurt"])),

    "dict_umls_precision": float(np.nanmean(df_metrics["dict_umls_precision"])),
    "dict_umls_recall": float(np.nanmean(df_metrics["dict_umls_recall"])),
    "dict_umls_f1": float(np.nanmean(df_metrics["dict_umls_f1"])),

    "ref_concept_detection_rate": float(df_metrics["umls_ref_cuis_extracted"].fillna("").astype(str).str.strip().ne("").mean()),
    "pred_concept_detection_rate": float(df_metrics["umls_pred_cuis_extracted"].fillna("").astype(str).str.strip().ne("").mean()),

    "per_sample_file": str(per_sample_path)
}

summary_df = pd.DataFrame([summary])

summary_path = METRIC_OUTPUT_DIR / "exp5_image_only_valid3000_three_additional_metrics_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nSaved per-sample metrics to:")
print(per_sample_path)

print("\nSaved summary to:")
print(summary_path)

print("\nExp5 additional metrics summary:")
display(summary_df)

Starting additional metrics for Exp5...

Computing Dictionary-based UMLS Concept F1...


Fast UMLS F1:   0%|          | 0/3000 [00:00<?, ?it/s]


Computing BERTScore...
calculating scores...
computing bert embedding.


  0%|          | 0/1284 [00:00<?, ?it/s]

computing greedy matching.


...

done in 117.80 seconds, 25.47 sentences/sec

Computing BLEURT...


BLEURT:   0%|          | 0/188 [00:00<?, ?it/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.



Saved per-sample metrics to:
/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/additional_metrics/exp5_image_only_valid3000_three_additional_metrics_per_sample.csv

Saved summary to:
/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp5_image_only_biomedclip_visual_prefix_t5/additional_metrics/exp5_image_only_valid3000_three_additional_metrics_summary.csv

Exp5 additional metrics summary:


,experiment,setting,n_samples,bertscore_precision,bertscore_recall,bertscore_f1,bleurt,dict_umls_precision,dict_umls_recall,dict_umls_f1,ref_concept_detection_rate,pred_concept_detection_rate,per_sample_file
0,Exp5_Image_only_BiomedCLIP_VisualPrefix_Exp4T5,Image only,3000,0.642882,0.577551,0.604346,0.338481,0.372761,0.464884,0.352147,0.822,0.925333,/content/drive/MyDrive/Senior2_Medical_Caption...
